In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Fausto\AppData\Local\Python\pythoncore-3.14-64\python.exe
3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]


In [2]:
import duckdb
print("duckdb ok")

duckdb ok


In [3]:
import duckdb

con = duckdb.connect()

path = "C:/Users/Fausto/Downloads/train-part-1_extracted/train-part-1/*.parquet"

df = con.execute(f"""
SELECT *
FROM read_parquet('{path}')
LIMIT 5
""").df()

df

,datetime,ui_language,eligible_templates,history,selected_template,session_end_completed
0,0.153461,en,"[G, E, B, A, K, H, J, L, F, D]","[{'template': 'A', 'n_days': 28.19564819335937...",B,False
1,2.827303,es,"[G, E, B, A, K, H, J, L, F, D]","[{'template': 'A', 'n_days': 29.836181640625},...",A,True
2,2.792662,en,"[G, E, B, K, H, J, L, F, D]","[{'template': 'G', 'n_days': 8.197543144226074...",J,True
3,4.904225,en,"[G, E, B, A, K, H, J, L, F, D]","[{'template': 'B', 'n_days': 29.00238037109375...",L,False
4,9.538715,en,"[K, H, G, E, B, J, L, F, D, A]","[{'template': 'B', 'n_days': 27.97145843505859...",B,True


1) Hoe big is this dataset?

In [4]:
con.execute(f"""
SELECT COUNT(*) AS n_rows
FROM read_parquet('{path}')
""").df()

,n_rows
0,25613243


2) What is the global sucess rate (reward)?

In [5]:
con.execute(f"""
SELECT AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
""").df()

,reward_rate
0,0.178333


3) How often occurs every template + how good does it work?

In [6]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate
0,C,999733,0.417601
1,A,1114900,0.306668
2,K,2635965,0.164083
3,L,2605053,0.163777
4,D,2603050,0.162900
5,J,2604892,0.162732
6,G,2604053,0.162057
7,F,2603232,0.161261
8,H,2635504,0.160798
9,E,2603496,0.160710


4) Does it differ per language?

In [7]:
con.execute(f"""
SELECT
  ui_language,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
GROUP BY ui_language
ORDER BY n DESC
""").df()

,ui_language,n,reward_rate
0,en,11109120,0.193555
1,es,6114460,0.149147
2,pt,2413486,0.147445
3,ru,1105242,0.181860
4,fr,970900,0.189121
5,de,811141,0.230966
6,it,456130,0.188284
7,vi,362143,0.190342
8,ar,335328,0.085367
9,pl,297297,0.234005


5) Important for bandit: how many templates are eligible per event?

In [8]:
con.execute(f"""
SELECT
  AVG(LEN(eligible_templates)) AS avg_eligible,
  MIN(LEN(eligible_templates)) AS min_eligible,
  MAX(LEN(eligible_templates)) AS max_eligible
FROM read_parquet('{path}')
""").df()

,avg_eligible,min_eligible,max_eligible
0,9.094416,1,10


How often is C available?

In [9]:
con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN 'C' IN eligible_templates THEN 1 ELSE 0 END) AS c_available
FROM read_parquet('{path}')
""").df()

,total,c_available
0,25613243,999733.0


What if we always choose C when C is available, 
and otherwise the best of the rest?

So firstly we must know:
What is the best template without C?

In [10]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
WHERE 'C' NOT IN eligible_templates
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate
0,A,1114900,0.306668
1,K,2635965,0.164083
2,L,2605053,0.163777
3,D,2603050,0.162900
4,J,2604892,0.162732
5,G,2604053,0.162057
6,F,2603232,0.161261
7,H,2635504,0.160798
8,E,2603496,0.160710
9,B,2603365,0.160256


There is a clear hierarchy:
C-> best overall(41.8%)
A -> best if C is not available (30.7%)
Rest -> all around 16%

This is no subtle difference.
A strong rule-based policy would be:
If C available -> choose C
Else -> choose A

1) How often would our new policy choose C vs A?

In [11]:
con.execute(f"""
SELECT
  COUNT(*) AS total,
  SUM(CASE WHEN 'C' IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_C,
  SUM(CASE WHEN 'C' NOT IN eligible_templates AND 'A' IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_A,
  SUM(CASE WHEN 'C' NOT IN eligible_templates AND 'A' NOT IN eligible_templates THEN 1 ELSE 0 END) AS policy_choose_neither
FROM read_parquet('{path}')
""").df()

,total,policy_choose_C,policy_choose_A,policy_choose_neither
0,25613243,999733.0,11004473.0,13609037.0


2) Off-policy estimate: reward of policy “C else A”

In [12]:
con.execute(f"""
WITH data AS (
  SELECT
    session_end_completed,
    selected_template,
    CASE 
      WHEN 'C' IN eligible_templates THEN 'C'
      ELSE 'A'
    END AS policy_template
  FROM read_parquet('{path}')
)
SELECT
  COUNT(*) AS total_events,
  SUM(CASE WHEN selected_template = policy_template THEN 1 ELSE 0 END) AS matched_events,
  AVG(CASE WHEN selected_template = policy_template THEN CASE WHEN session_end_completed THEN 1 ELSE 0 END END) AS estimated_policy_reward
FROM data
""").df()

,total_events,matched_events,estimated_policy_reward
0,25613243,2114633.0,0.359114


What do we see?
Total events
2,561,324
Matched events
2,114,633 (useful for evaluation)
-> That's ~82% coverage
That's good for off-policy evaluation.
Estimated policy reward
0.3591 (35.9%)
Comparison with current logging policy(=In the historical data, Duolingo selected a template at random from the eligible pool, with equal probability for each template. That random decision process is what generated the dataset — that’s the logging policy)
Logging policy reward was:
17.8%
Our simple new rule:
If C available -> C
Else -> A
35.9%

A matched event is:
An event where the logging policy chose by chance exactly the same template as your new policy would have chosen.
Why do we only use matched events?

!!!Because we only know the reward of what was actually shown!!!

For example:
Suppose:
Eligible = {C, D, E}
Logging chose D
Our new policy would have chosen C
Then we know:
Reward of D -> yes
Reward of C -> no (counterfactual unknown)
So we can't use that event to evaluate C.

This compares C vs A in exactly the same context/set/pool
It removes a big part of the selection bias.

In [15]:
con.execute(f"""
SELECT
  selected_template,
  COUNT(*) AS n,
  AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
FROM read_parquet('{path}')
WHERE 'C' IN eligible_templates
  AND 'A' IN eligible_templates
GROUP BY selected_template
ORDER BY reward_rate DESC
""").df()

,selected_template,n,reward_rate


What does this mean in terms of content?

!!!C and A apparently never occur in the same eligible pool!!!

So:
!!!C is shown in a completely different segment than A,
this confirms selection bias!!!
You can't compare C and A within the same context.

The rule:
If C available -> C
Else -> A
actually works because:
C-segment = high engagement users
A-segment = other users
So your policy is actually:
If user in C-segment -> choose C
Otherwise -> choose A
But that segment difference is already incorporated in the data.

1) Which eligible pools occur the most?

In [16]:
con.execute(f"""
SELECT
  eligible_templates,
  COUNT(*) AS n
FROM read_parquet('{path}')
GROUP BY eligible_templates
ORDER BY n DESC
LIMIT 20
""").df()

,eligible_templates,n
0,"[G, E, B, K, H, J, L, F, D]",7485674
1,"[G, E, B, A, K, H, J, L, F, D]",6778260
2,"[K, H, G, E, B, J, L, F, D]",6100955
3,"[K, H, G, E, B, J, L, F, D, A]",4164577
4,[C],999733
5,"[A, K, H]",38181
6,"[K, H, A]",23455
7,"[K, H]",22408


2) Within every pool: reward per selected_template (and top/best template per pool)

In [18]:
con.execute(f"""
WITH stats AS (
  SELECT
    eligible_templates,
    selected_template,
    COUNT(*) AS n,
    AVG(CASE WHEN session_end_completed THEN 1 ELSE 0 END) AS reward_rate
  FROM read_parquet('{path}')
  GROUP BY eligible_templates, selected_template
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY eligible_templates
      ORDER BY reward_rate DESC
    ) AS rnk
  FROM stats
  WHERE n >= 5000
)
SELECT
  eligible_templates,
  selected_template AS best_template,
  n,
  reward_rate
FROM ranked
WHERE rnk = 1
ORDER BY n DESC
LIMIT 50
""").df()

,eligible_templates,best_template,n,reward_rate
0,[C],C,999733,0.417601
1,"[G, E, B, K, H, J, L, F, D]",D,832366,0.062925
2,"[G, E, B, A, K, H, J, L, F, D]",L,679120,0.311706
3,"[K, H, G, E, B, J, L, F, D]",L,678053,0.055658
4,"[K, H, G, E, B, J, L, F, D, A]",L,416585,0.300299
5,"[A, K, H]",A,12661,0.456046
6,"[K, H]",H,11127,0.103622
7,"[K, H, A]",A,7811,0.439508


What do we see here?
There are a few big eligible pools (million events).
The best template is not always C within these pools.
In some pools wins L.
In little pools (such as [A, K, H]) wins A with a high reward rate.
C is actually in its own pool= [C].
That means:
Templates live in different worlds.
Reward rate is strongly dependent of the eligible pool.

In [ ]:
Logistic Model:

STEP1 : load the data (fast + safe) with DuckDB in Python

In [4]:
import duckdb
import pandas as pd

# 1) Connect
con = duckdb.connect()

# 2) Zet hier je pad naar de parquet files (training of test)
# Voorbeeld: "data/training/*.parquet" of "/path/to/training/*.parquet"
PARQUET_GLOB = "C:/Users/Fausto/Downloads/train-part-1_extracted/train-part-1/*.parquet"

# 3) Maak een view (handig om later queries te hergebruiken)
con.execute(f"""
CREATE OR REPLACE VIEW notif AS
SELECT *
FROM read_parquet('{PARQUET_GLOB}');
""")

# 4) Snelle sanity checks
print("Row count (approx query):")
print(con.execute("SELECT COUNT(*) AS n FROM notif").fetchdf())

print("\nColumns + types:")
print(con.execute("DESCRIBE notif").fetchdf())

print("\nOverall reward rate:")
print(con.execute("""
SELECT AVG(CAST(session_end_completed AS INTEGER)) AS reward_rate
FROM notif
""").fetchdf())

print("\nExample rows:")
print(con.execute("""
SELECT datetime, ui_language, eligible_templates, history, selected_template, session_end_completed
FROM notif
LIMIT 3
""").fetchdf())

Row count (approx query):
          n
0  25613243

Columns + types:
             column_name                                 column_type null  \
0               datetime                                      DOUBLE  YES   
1            ui_language                                     VARCHAR  YES   
2     eligible_templates                                   VARCHAR[]  YES   
3                history  STRUCT("template" VARCHAR, n_days FLOAT)[]  YES   
4      selected_template                                     VARCHAR  YES   
5  session_end_completed                                     BOOLEAN  YES   

    key default extra  
0  None    None  None  
1  None    None  None  
2  None    None  None  
3  None    None  None  
4  None    None  None  
5  None    None  None  

Overall reward rate:
   reward_rate
0     0.178333

Example rows:
   datetime ui_language              eligible_templates  \
0  0.153461          en  [G, E, B, A, K, H, J, L, F, D]   
1  2.827303          es  [G, E, B, A, K

Step 2: drawing a training sample + prepare features
We want:
Only real decision points (len(eligible_templates) >= 2)
A sample of ±1 miljoen rows (for speed)

In [5]:
query = """
SELECT
    ui_language,
    selected_template,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward
FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 1000000 ROWS
"""

df = con.execute(query).df()

print(df.shape)
print(df.head())
print("Reward rate in sample:", df['reward'].mean())

(961183, 4)
  ui_language selected_template  n_eligible  reward
0          en                 F           9       0
1          en                 K           9       0
2          en                 B          10       0
3          en                 B          10       0
4          en                 J          10       1
Reward rate in sample: 0.16851941825854183


Step 3 — Logistic Regression training

In [6]:
!pip install scikit-learn

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

X = df[['ui_language', 'selected_template', 'n_eligible']]
y = df['reward']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'),
         ['ui_language', 'selected_template']),
        ('num', 'passthrough', ['n_eligible'])
    ]
)

model = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)

print("AUC:", auc)

AUC: 0.7250913824626937


Wat does 0.73 AUC mean concrete?
0.5 = random guessing
0.6 = poor signal
0.7 = solidly predictive
0.8+ = strong model
So:
There is a clear structure in the data.
!!!Template + language really matters!!!

Stap 4A — Check template ranking volgens model

In [15]:
import numpy as np

# Extract coefficients
feature_names = model.named_steps['preprocess'].get_feature_names_out()
coefs = model.named_steps['clf'].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs
})

# Kijk enkel naar template effecten
template_effects = coef_df[coef_df["feature"].str.contains("selected_template")]

template_effects.sort_values("coef", ascending=False).head(10)

,feature,coef
23,cat__selected_template_A,-1.314053
27,cat__selected_template_F,-1.421965
30,cat__selected_template_J,-1.427136
29,cat__selected_template_H,-1.429824
32,cat__selected_template_L,-1.436231
31,cat__selected_template_K,-1.450668
24,cat__selected_template_B,-1.451834
25,cat__selected_template_D,-1.456089
26,cat__selected_template_E,-1.461394
28,cat__selected_template_G,-1.463152


What we want to test now...
We want to test:
If we choose the template with the highest predicted probability within every eligible pool,
do we receive a higher reward than random?

That's the core.
But here is the problem
In your current dataset you have only:
selected_template

!!!Because we only know the reward of what was actually shown.
You don't know what the reward would have been for the other templates in that pool.
So we can't simulate perfectly.!!!

But...
Because logging policy was uniform random (implies selection bias)
we can do a correct off-policy evaluation.

Goal:

Simulate what would have happened if we choose the template with the highest predicted probability within every pool.

Stap 4 — Model-based policy simulation

Stap 4A — New sample with eligible_templates

In [16]:
query = """
SELECT
    ui_language,
    selected_template,
    eligible_templates,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward
FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 300000 ROWS
"""

df_policy = con.execute(query).df()
print(df_policy.shape)

(288313, 5)


Stap 4B — Policy simulation

In [17]:
import pandas as pd
import numpy as np

# df_policy moet deze kolommen hebben:
# ['ui_language', 'selected_template', 'eligible_templates', 'n_eligible', 'reward']

# 1) Geef elke rij een id zodat we later kunnen groeperen
df_policy = df_policy.reset_index(drop=True)
df_policy['row_id'] = np.arange(len(df_policy))

# 2) Explode eligible_templates -> long format
long = df_policy[['row_id', 'ui_language', 'n_eligible', 'reward', 'selected_template', 'eligible_templates']].explode('eligible_templates')
long = long.rename(columns={'eligible_templates': 'cand_template'})

# 3) Predict in batch: we zetten cand_template tijdelijk in de kolomnaam die het model verwacht
X_long = long[['ui_language', 'n_eligible']].copy()
X_long['selected_template'] = long['cand_template'].astype(str)

# batch predict
long['p_hat'] = model.predict_proba(X_long)[:, 1]

# 4) Kies beste template per row_id
best_idx = long.groupby('row_id')['p_hat'].idxmax()
chosen = long.loc[best_idx, ['row_id', 'cand_template']].rename(columns={'cand_template': 'model_choice'})

# 5) Join terug op df_policy
df_eval = df_policy.merge(chosen, on='row_id', how='left')

# 6) Matched evaluation
matched = df_eval[df_eval['model_choice'] == df_eval['selected_template']]

matched_fraction = len(matched) / len(df_eval)
reward_matched = matched['reward'].mean()
reward_baseline = df_eval['reward'].mean()

print("Matched fraction:", matched_fraction)
print("Reward in matched cases:", reward_matched)
print("Random baseline reward:", reward_baseline)

# Uplift (absolute en relatief)
print("Absolute uplift:", reward_matched - reward_baseline)
print("Relative uplift:", (reward_matched / reward_baseline) - 1)

Matched fraction: 0.10611020757724479
Reward in matched cases: 0.16822689230107557
Random baseline reward: 0.17418403163067805
Absolute uplift: -0.005957139329602484
Relative uplift: -0.03420026091848305


Stap 5 — IPS evaluatie (copy-paste)

In [40]:
# IPS evaluation for deterministic policy "choose model_choice"
# Assumption: logging was uniform random over eligible_templates
# => b(a|t) = 1 / n_eligible  -> weight = n_eligible

df_eval['match'] = (df_eval['model_choice'] == df_eval['selected_template']).astype(int)
df_eval['w'] = df_eval['n_eligible']  # 1 / (1/n_eligible)

ips_estimate = (df_eval['match'] * df_eval['reward'] * df_eval['w']).mean()

baseline = df_eval['reward'].mean()

print("IPS estimated reward of model-policy:", ips_estimate)
print("Random baseline reward:", baseline)
print("Absolute uplift:", ips_estimate - baseline)
print("Relative uplift:", ips_estimate / baseline - 1)

IPS estimated reward of model-policy: 0.1741415758350459
Random baseline reward: 0.1741415758350459
Absolute uplift: 0.0
Relative uplift: 0.0


Let's add recency (= how many days ago a template was last shown to the user) and after that again:

model training

policy simulation (vectorized)

IPS evaluation

1) Sample with recency + eligible_templates

In [9]:
query = """
SELECT
    ui_language,
    selected_template,
    eligible_templates,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward,

    (
      SELECT min(h.n_days)
      FROM unnest(history) AS u(h)
      WHERE h.template = selected_template
    ) AS days_since_last_seen

FROM notif
WHERE array_length(eligible_templates) >= 2
USING SAMPLE 300000 ROWS
"""
df = con.execute(query).df()

print(df.shape)
print("Missing recency:", df['days_since_last_seen'].isna().mean())
print(df.head())

(288329, 6)
Missing recency: 0.4257670924534126
  ui_language selected_template              eligible_templates  n_eligible  \
0          en                 F     [G, E, B, K, H, J, L, F, D]           9   
1          tr                 J  [G, E, B, A, K, H, J, L, F, D]          10   
2          en                 J     [G, E, B, K, H, J, L, F, D]           9   
3          tr                 E     [G, E, B, K, H, J, L, F, D]           9   
4          en                 F     [G, E, B, K, H, J, L, F, D]           9   

   reward  days_since_last_seen  
0       0              8.566932  
1       0                   NaN  
2       0              8.100726  
3       0                   NaN  
4       0             15.550262  


Next stap (Stap 2) — preparing recency effect

In [10]:
df['days_since_last_seen'] = df['days_since_last_seen'].fillna(999).astype(float)

print(df['days_since_last_seen'].describe())
print("Share 'never' (999):", (df['days_since_last_seen'] == 999).mean())

count    288329.000000
mean        430.158233
std         489.847321
min           0.003463
25%           4.999792
50%          17.901808
75%         999.000000
max         999.000000
Name: days_since_last_seen, dtype: float64
Share 'never' (999): 0.4257670924534126


After that: again logistic regression training (with recency effect)

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

X = df[['ui_language', 'selected_template', 'n_eligible', 'days_since_last_seen']]
y = df['reward']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'),
         ['ui_language', 'selected_template']),
        ('num', 'passthrough', ['n_eligible', 'days_since_last_seen'])
    ]
)

model_rec = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

model_rec.fit(X_train, y_train)

y_pred = model_rec.predict_proba(X_test)[:, 1]
print("AUC with recency:", roc_auc_score(y_test, y_pred))

AUC with recency: 0.7416510853819237


c:\Users\Fausto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [12]:
query_long = """
WITH base AS (
  SELECT
    row_number() OVER () - 1 AS row_id,
    ui_language,
    selected_template,
    eligible_templates,
    array_length(eligible_templates) AS n_eligible,
    CAST(session_end_completed AS INTEGER) AS reward,
    history
  FROM notif
  WHERE array_length(eligible_templates) >= 2
  USING SAMPLE 200000 ROWS
),
cand AS (
  SELECT
    b.row_id,
    b.ui_language,
    b.n_eligible,
    b.reward,
    b.selected_template,
    t AS cand_template,
    (
      SELECT min(h.n_days)
      FROM unnest(b.history) AS u(h)
      WHERE h.template = t
    ) AS days_since_last_seen_cand
  FROM base b
  CROSS JOIN unnest(b.eligible_templates) AS u(t)
)
SELECT *
FROM cand;
"""

long = con.execute(query_long).df()
print(long.shape)
print("Missing candidate recency:", long['days_since_last_seen_cand'].isna().mean())

(1810048, 7)
Missing candidate recency: 0.4287344865992504


It means:
Almost half of candidate templates are “new” (or unseen recently)
That novelty signal exists but even using it, greedy didn’t improve IPS reward